In [18]:
import pandas as pd
import random
import nltk
from nltk.corpus import wordnet

# Download necessary NLTK data
nltk.download("wordnet")
nltk.download("omw-1.4")

df1 = pd.read_csv("10k_phrases_batch1.csv")
df2 = pd.read_csv("8k_batch2.csv")

[nltk_data] Downloading package wordnet to
[nltk_data]     /home/ritisha21089/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /home/ritisha21089/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [19]:
df2['Labels_List'] = df2['Labels'].apply(eval)  # Convert string representation to list

labels_of_interest = ['PREPROCESSING']

# Filter rows containing any of the specified labels
filtered_df = df2[df2['Labels_List'].apply(lambda x: any(label in x for label in labels_of_interest))]

# Display the filtered rows
print(filtered_df)

      Unnamed: 0                                 Ingredient_Phrases Quantity  \
0              0  1 large salmon fillet , skin and small pin bon...        1   
3              3                10 cups warm freshly popped popcorn       10   
5              5  2 1/2 lbs boneless skinless chicken breasts , ...    2 1/2   
6              6  1 package phyllo dough , thawed -LRB- usually ...        1   
9              9  1 chipotle chile -LRB- Please note that I opte...        1   
...          ...                                                ...      ...   
8003        8003  1 cup grated cheddar cheese or 1 cup monterey ...        1   
8006        8006  6 dried red chili peppers , , cut and seeds re...        6   
8007        8007     100 g fresh shiitake mushrooms , sliced thinly      100   
8008        8008         75 g ready-to-eat dried apricots , chopped       75   
8010        8010  shredded sharp cheddar cheese or monterey jack...      NaN   

         Unit        Ingredient Name   

In [23]:
filtered_df = filtered_df.drop('Labels_List', axis=1) 

In [24]:
filtered_df.head()

,Ingredient_Phrases,Quantity,Unit,Ingredient Name,Form,State,Dry/Fresh,Size,Preprocessing
0,"1 large salmon fillet , skin and small pin bon...",1,NaN,salmon fillet,NaN,NaN,NaN,NaN,skin and small pin bones removed
3,10 cups warm freshly popped popcorn,10,cups,popcorn,NaN,NaN,fresh,NaN,popped freshly
5,"2 1/2 lbs boneless skinless chicken breasts , ...",2 1/2,lbs,chicken breasts,NaN,NaN,NaN,1 inch,"cubed, boneless, skinless"
6,"1 package phyllo dough , thawed -LRB- usually ...",1,package,phyllo dough,NaN,NaN,NaN,NaN,thawed
9,1 chipotle chile -LRB- Please note that I opte...,1,NaN,chipotle chile,ground,NaN,NaN,NaN,toasted


In [33]:
df = filtered_df.copy()

In [34]:
df.head()

,Ingredient_Phrases,Quantity,Unit,Ingredient Name,Form,State,Dry/Fresh,Size,Preprocessing
0,"1 large salmon fillet , skin and small pin bon...",1,NaN,salmon fillet,NaN,NaN,NaN,NaN,skin and small pin bones removed
3,10 cups warm freshly popped popcorn,10,cups,popcorn,NaN,NaN,fresh,NaN,popped freshly
5,"2 1/2 lbs boneless skinless chicken breasts , ...",2 1/2,lbs,chicken breasts,NaN,NaN,NaN,1 inch,"cubed, boneless, skinless"
6,"1 package phyllo dough , thawed -LRB- usually ...",1,package,phyllo dough,NaN,NaN,NaN,NaN,thawed
9,1 chipotle chile -LRB- Please note that I opte...,1,NaN,chipotle chile,ground,NaN,NaN,NaN,toasted


In [13]:
# Assuming the DataFrame is named 'df'

# Access the second row (index = 1)
second_row = filtered_df.iloc[1]

# Print each column and its value
for column, value in second_row.items():
    print(f"{column}: {value}")

Ingredient_Phrases: 10 cups warm freshly popped popcorn
Quantity: 10
Unit: cups
Ingredient Name: popcorn
Form: nan
State: nan
Dry/Fresh: fresh
Size: nan
Preprocessing: popped freshly
Labels: ['QUANTITY', 'UNIT', 'O', 'PREPROCESSING', 'PREPROCESSING', 'NAME']


In [14]:
!pip install fuzzywuzzy

In [35]:
from fuzzywuzzy import process

def update_column_with_similarity(row, column_name, reference_column):
    """
    Update a column's value with the most similar word/phrase from the reference column.
    """
    reference_text = row[reference_column]
    original_value = row[column_name]
    
    if pd.isnull(original_value) or pd.isnull(reference_text):
        return original_value  # No change if either value is missing

    # Tokenize the reference text and find the most similar match for the column value
    tokens = reference_text.split()
    best_match, score = process.extractOne(original_value, tokens)
    
    # Update the column value if the similarity score is high (e.g., above 80)
    if score > 80:
        return best_match
    return original_value

# Apply this function to your DataFrame
for col in ['Form', 'State', 'Dry/Fresh']:
    filtered_df[col] = filtered_df.apply(lambda row: update_column_with_similarity(row, col, 'Ingredient_Phrases'), axis=1)

In [36]:
def filter_preprocessing(row):
    """
    Filters the preprocessing column to only include values found in the Ingredient_Phrases.
    """
    ingredient_phrase = row["Ingredient_Phrases"]
    preprocessing_values = row["Preprocessing"].split(", ")  # Split preprocessing values by comma

    # Check for the presence of each value in the ingredient phrase
    filtered_values = [value for value in preprocessing_values if value in ingredient_phrase]

    # Return the filtered values as a comma-separated string
    return ", ".join(filtered_values)

# Apply the function to the DataFrame
filtered_df["Preprocessing"] = filtered_df.apply(filter_preprocessing, axis=1)

In [37]:
filtered_df.head()

,Ingredient_Phrases,Quantity,Unit,Ingredient Name,Form,State,Dry/Fresh,Size,Preprocessing
0,"1 large salmon fillet , skin and small pin bon...",1,NaN,salmon fillet,NaN,NaN,NaN,NaN,skin and small pin bones removed
3,10 cups warm freshly popped popcorn,10,cups,popcorn,NaN,NaN,freshly,NaN,
5,"2 1/2 lbs boneless skinless chicken breasts , ...",2 1/2,lbs,chicken breasts,NaN,NaN,NaN,1 inch,"boneless, skinless"
6,"1 package phyllo dough , thawed -LRB- usually ...",1,package,phyllo dough,NaN,NaN,NaN,NaN,thawed
9,1 chipotle chile -LRB- Please note that I opte...,1,NaN,chipotle chile,ground,NaN,NaN,NaN,


In [17]:
filtered_df.to_csv('preprocessing_new.csv', index=False)